# MethodThinker 自动化训练 Notebook

> **参数化训练模板** - 支持papermill参数注入

**自动化执行**:
```bash
python scripts/colab_automation.py run --preset quick
```

**参数说明**:
- `BASE_MODEL`: 基座模型名称
- `NUM_EPOCHS`: 训练轮数
- `BATCH_SIZE`: 批大小
- `LEARNING_RATE`: 学习率
- `MAX_LENGTH`: 最大序列长度
- `LORA_R`: LoRA秩
- `LORA_ALPHA`: LoRA alpha
- `HF_MIRROR`: HuggingFace镜像地址
- `MAX_SAMPLES`: 最大训练样本数
- `OUTPUT_DIR`: 输出目录
- `RUN_TIMESTAMP`: 运行时间戳

---

## 训练参数配置

以下参数可通过papermill注入覆盖。

In [ ]:
# Papermill参数 - 可通过命令行覆盖
BASE_MODEL = "Qwen/Qwen2.5-Math-1.5B"
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 5e-5
MAX_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
HF_MIRROR = "https://hf-mirror.com"
MAX_SAMPLES = 1000
OUTPUT_DIR = "outputs/colab_training"
RUN_TIMESTAMP = ""

# 内部变量
TRAINING_TIME = 0
FINAL_LOSS = 0
TRAINING_STATUS = "pending"

print("参数配置完成")
print(f"基座模型: {BASE_MODEL}")
print(f"训练轮数: {NUM_EPOCHS}")
print(f"批大小: {BATCH_SIZE}")
print(f"学习率: {LEARNING_RATE}")
print(f"最大长度: {MAX_LENGTH}")
print(f"LoRA秩: {LORA_R}")
print(f"样本数: {MAX_SAMPLES}")

## 1. GPU环境检查

验证GPU状态，确保T4 GPU已启用。

In [ ]:
# GPU状态检查
import torch
import sys

print("="*50)
print("GPU环境检查")
print("="*50)

gpu_available = torch.cuda.is_available()

if not gpu_available:
    print("GPU未启用！")
    print("请执行以下步骤：")
    print("  1. 点击菜单 '运行时' → '更改运行时类型'")
    print("  2. 选择 'T4 GPU'")
    print("  3. 点击 '保存' 重新连接")
    TRAINING_STATUS = "error_no_gpu"
else:
    print(f"GPU已启用")
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"显存大小: {gpu_mem:.2f} GB")
    print(f"CUDA版本: {torch.version.cuda}")
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f"支持BF16: {bf16_supported}")

print("="*50)

# 记录环境信息
ENV_INFO = {
    "gpu_available": gpu_available,
    "gpu_name": torch.cuda.get_device_name(0) if gpu_available else None,
    "gpu_memory_gb": round(gpu_mem, 2) if gpu_available else None,
    "cuda_version": torch.version.cuda if gpu_available else None,
    "bf16_supported": bf16_supported if gpu_available else False
}

In [ ]:
# 详细GPU信息
!nvidia-smi

## 2. 环境设置

设置HuggingFace镜像并安装依赖。

In [ ]:
# 设置HuggingFace镜像
import os

os.environ['HF_ENDPOINT'] = HF_MIRROR
print(f"HuggingFace镜像已设置: {HF_MIRROR}")

# 安装核心依赖
!pip install -q torch>=2.1.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.40.0 accelerate>=0.25.0 peft>=0.7.0 trl>=0.7.0 datasets>=2.14.0
!pip install -q pyyaml pydantic numpy scipy sympy pandas tqdm rich python-dotenv papermill

print("依赖已安装")

In [ ]:
# 环境验证
import torch
import transformers
import accelerate
import peft
import trl

print("="*50)
print("环境验证报告")
print("="*50)
print(f"PyTorch版本: {torch.__version__}")
print(f"Transformers版本: {transformers.__version__}")
print(f"Accelerate版本: {accelerate.__version__}")
print(f"PEFT版本: {peft.__version__}")
print(f"TRL版本: {trl.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("="*50)
print("环境验证完成")

## 3. Google Drive挂载

挂载Drive用于保存模型和结果。

In [ ]:
# 挂载Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    SAVE_DIR = '/content/drive/MyDrive/method-thinker-models'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Drive已挂载，模型将保存到: {SAVE_DIR}")
except ImportError:
    print("不在Colab环境，跳过Drive挂载")
    DRIVE_MOUNTED = False
    SAVE_DIR = OUTPUT_DIR

## 4. 模型加载

加载基座模型和分词器。

In [ ]:
# 加载基座模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"加载模型: {BASE_MODEL}")
print("这可能需要几分钟...")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# 设置pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("模型加载完成")

# 显示模型信息
model_params = model.num_parameters() / 1e9
print(f"模型参数量: {model_params:.2f}B")
print(f"模型设备: {model.device}")

## 5. LoRA配置

应用LoRA实现高效微调。

In [ ]:
# 应用LoRA配置
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)

# 显示可训练参数
model.print_trainable_parameters()

print(f"LoRA配置已应用: r={LORA_R}, alpha={LORA_ALPHA}")

## 6. 数据准备

加载或创建训练数据。

In [ ]:
# 准备训练数据
import json
import os
from datasets import Dataset

TRAIN_PATH = "data/train_data/train.json"

# 检查是否存在训练数据
if os.path.exists(TRAIN_PATH):
    with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print(f"加载训练数据: {len(raw_data)} 条")
else:
    # 创建示例数据
    print("创建示例训练数据...")
    os.makedirs("data/train_data", exist_ok=True)
    
    sample_data = [
        {
            "problem_id": f"sample_{i:03d}",
            "problem": f"示例数学问题 {i}",
            "problem_type": "代数",
            "difficulty": 1,
            "candidate_methods": [
                {"method_name": "直接计算", "applicability_score": 0.8},
                {"method_name": "代入验证", "applicability_score": 0.7}
            ],
            "selected_method": "直接计算",
            "selection_reasoning": "问题适合直接计算求解",
            "solution_steps": ["步骤1: 分析问题", "步骤2: 计算", "步骤3: 验证答案"],
            "reflection": "解答正确，方法选择合理"
        }
        for i in range(min(MAX_SAMPLES, 100))
    ]
    
    with open(TRAIN_PATH, 'w', encoding='utf-8') as f:
        json.dump(sample_data, f, ensure_ascii=False, indent=2)
    
    # 创建验证数据
    with open("data/train_data/val.json", 'w', encoding='utf-8') as f:
        json.dump(sample_data[:10], f, ensure_ascii=False, indent=2)
    
    raw_data = sample_data
    print(f"示例数据已创建: {len(raw_data)} 条")

# 限制样本数
if MAX_SAMPLES and len(raw_data) > MAX_SAMPLES:
    raw_data = raw_data[:MAX_SAMPLES]
    print(f"样本数限制为: {MAX_SAMPLES}")

In [ ]:
# 格式化训练数据
def format_training_sample(sample):
    """格式化训练样本为对话格式"""
    candidates_str = "\n".join([
        f"{i+1}. {m['method_name']}（评分：{m['applicability_score']}）"
        for i, m in enumerate(sample['candidate_methods'])
    ])
    
    steps_str = "\n".join(sample['solution_steps'])
    
    input_text = f"""【问题】
{sample['problem']}

【题型】
{sample['problem_type']}

【候选方法】
{candidates_str}

请分析各方法并选择最合适的方法解答问题。
"""
    
    output_text = f"""【方法选择】
选中方法：{sample['selected_method']}

【选择理由】
{sample['selection_reasoning']}

【解答过程】
{steps_str}

【反思与验证】
{sample['reflection']}
"""
    
    return {
        'text': input_text + output_text,
        'input': input_text,
        'output': output_text
    }

# 格式化数据
formatted_data = [format_training_sample(s) for s in raw_data]
train_dataset = Dataset.from_list(formatted_data)

print(f"数据集已准备: {len(train_dataset)} 条")
print(f"\n示例数据（第一条）:")
print(train_dataset[0]['text'][:500] + "...")

## 7. 训练执行

启动方法论注入训练。

In [ ]:
# 配置训练参数
from transformers import TrainingArguments

# 计算梯度累积步数以保持有效batch=32
gradient_accumulation_steps = max(1, 32 // BATCH_SIZE)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=50,
    save_steps=200,
    save_total_limit=3,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    gradient_checkpointing=True,
    eval_strategy="no",
    seed=42,
    report_to="none",
)

print("训练参数已配置")
print(f"\n训练配置:")
print(f"  - 训练轮数: {training_args.num_train_epochs}")
print(f"  - 批大小: {training_args.per_device_train_batch_size}")
print(f"  - 有效批大小: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  - 学习率: {training_args.learning_rate}")
print(f"  - BF16训练: {training_args.bf16}")

In [ ]:
# 开始训练
from trl import SFTTrainer
import time

print("="*50)
print("开始方法论注入训练")
print("="*50)

# 创建训练器
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    max_seq_length=MAX_LENGTH,
    packing=False,
)

# 记录开始时间
start_time = time.time()
TRAINING_STATUS = "running"

# 开始训练
try:
    train_result = trainer.train()
    elapsed_time = time.time() - start_time
    TRAINING_TIME = elapsed_time
    FINAL_LOSS = train_result.training_loss
    TRAINING_STATUS = "completed"
    
    print("\n" + "="*50)
    print("训练完成")
    print("="*50)
    print(f"训练耗时: {elapsed_time:.1f} 秒")
    print(f"最终损失: {train_result.training_loss:.4f}")
    print(f"训练步数: {train_result.global_step}")
except Exception as e:
    TRAINING_STATUS = "error"
    TRAINING_ERROR = str(e)
    print(f"训练失败: {e}")
    raise

## 8. 模型保存

保存训练后的模型到本地和Drive。

In [ ]:
# 保存模型
import shutil
from datetime import datetime

# 本地保存
local_path = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(local_path)
tokenizer.save_pretrained(local_path)

print(f"模型已保存到本地: {local_path}")

# Drive保存
if DRIVE_MOUNTED:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    if RUN_TIMESTAMP:
        timestamp = RUN_TIMESTAMP
    drive_path = f"{SAVE_DIR}/model_{timestamp}"
    shutil.copytree(local_path, drive_path)
    print(f"模型已保存到Drive: {drive_path}")
else:
    drive_path = local_path

In [ ]:
# 保存训练报告
training_report = {
    "timestamp": datetime.now().isoformat(),
    "status": TRAINING_STATUS,
    "training_time": TRAINING_TIME,
    "final_loss": FINAL_LOSS,
    "global_step": train_result.global_step if TRAINING_STATUS == "completed" else None,
    "config": {
        "base_model": BASE_MODEL,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "effective_batch_size": BATCH_SIZE * gradient_accumulation_steps,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "max_samples": MAX_SAMPLES
    },
    "environment": ENV_INFO,
    "model_path": drive_path
}

report_path = os.path.join(drive_path, "training_report.json")
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(training_report, f, ensure_ascii=False, indent=2)

print(f"训练报告已保存: {report_path}")

# 输出报告供papermill提取
print("\n" + "="*50)
print("TRAINING_REPORT_JSON")
print("="*50)
print(json.dumps(training_report, ensure_ascii=False))

## 9. 快速验证

测试模型生成效果。

In [ ]:
# 测试模型生成
test_problem = "求方程 x^2 - 4x + 3 = 0 的解"

input_text = f"""【问题】
{test_problem}

【题型】
二次方程

【候选方法】
1. 因式分解法（评分：0.9）
2. 公式法（评分：0.8）

请分析各方法并选择最合适的方法解答问题。
"""

# 编码输入
inputs = tokenizer(input_text, return_tensors="pt", max_length=MAX_LENGTH, truncation=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# 生成
print("生成解答...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# 解码
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 提取生成部分
if input_text in generated_text:
    generated_text = generated_text[len(input_text):]

print("\n" + "="*50)
print("模型生成结果")
print("="*50)
print(generated_text)

## 10. 结果汇总

输出最终训练结果。

In [ ]:
# 最终汇总
print("="*50)
print("训练总结")
print("="*50)
print(f"状态: {TRAINING_STATUS}")
print(f"基座模型: {BASE_MODEL}")
print(f"训练方法: LoRA (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"训练轮数: {NUM_EPOCHS}")
print(f"训练样本: {len(train_dataset)}")
if TRAINING_STATUS == "completed":
    print(f"训练耗时: {TRAINING_TIME:.1f} 秒")
    print(f"最终损失: {FINAL_LOSS:.4f}")
print(f"\n模型保存位置:")
print(f"  本地: {local_path}")
print(f"  Drive: {drive_path}")
print("="*50)

# 供自动化脚本读取的输出
print("\nFINAL_STATUS:", TRAINING_STATUS)
print("FINAL_MODEL_PATH:", drive_path)
print("FINAL_TRAINING_TIME:", TRAINING_TIME)
print("FINAL_LOSS:", FINAL_LOSS)
print("\n所有步骤完成！")